# Lesson 8.7: Make failures terminal, visible and recoverable

## What this lesson adds

Show an expired attempt losing permission to finish a job.

The matching written lesson is `modules/08-agents/07-failures-and-queues.md`. Read its conceptual and organisational argument alongside this practical activity. All customer records here are fictional. Live cells are a recommended part of technical study after API setup; the offline route remains useful for inspecting mechanics.

## Before running

Use the project `.venv` kernel. The setup cell below locates the course and reads the chosen learning mode. It does not send a provider request. If live mode is enabled, this notebook has a twelve-request gateway ceiling and a maximum of one thousand output tokens per response. The framework comparison also permits one SDK turn per invocation, whose usage is displayed separately. For billing, consult the actual ledger and current prices rather than assuming that a request count is a currency limit.


The included helper path is explained in `reference/application-code-walkthrough.md`. Open the section matching this activity before treating an imported function as understood.

## Business context: Harbour

If a program reports “sending” and then stops, Harbour needs to know whether anything was sent. A terminal outcome marks the end of a run, such as completion or failure. Use the case record to distinguish a confirmed result from an unknown result. Recovery may require checking what happened before trying again.

Defend the operating arrangement against missing evidence, stale state and an unaccepted handoff.

[Company and supply-chain context](../../HARBOUR.md) · [Module 8 business situation](../../reference/harbour-learning-journey.md#module-8)



In [ ]:
from ai_orgdesign.workspace import project_root, live_mode
ROOT = project_root()
RUN_LIVE = live_mode()
if RUN_LIVE:
    from ai_orgdesign.provider import Gateway
    gateway = Gateway(max_requests=12, max_output_tokens=1000)
print("Live mode:", RUN_LIVE)


## Execute the written lesson's mechanism

This cell runs the worked example from the lesson. Run it unchanged first. Look for the case or settings passed into the main function, then for `print` (what the notebook shows) or `assert` (a condition the example checks). This cell does not require a live request.

In [ ]:
from queue import Queue, Empty
from threading import Thread

def worker(job_id, operation, events):
    try:
        events.put({"job": job_id, "kind": "progress", "text": "Started"})
        result = operation()
        events.put({"job": job_id, "kind": "success", "result": result})
    except Exception as error:
        events.put({"job": job_id, "kind": "error", "error": type(error).__name__})

events = Queue()
thread = Thread(target=worker, args=("J-17", lambda: 12-10, events))
thread.start()
terminal = None
while terminal is None:
    try:
        event = events.get(timeout=2)
    except Empty:
        raise TimeoutError("No event arrived within the waiting policy")
    if event["kind"] in {"success", "error"}:
        terminal = event
thread.join(timeout=2)
assert terminal["result"] == 2


## Predict the intermediate result

Which attempt owns the work after time eleven?

Write or say an answer before running the next cell. Run it once as provided and read what appears below the cell. If it differs from the prediction, locate the input or rule that produced the result.


In [ ]:
from tempfile import TemporaryDirectory
from pathlib import Path
from ai_orgdesign.business import CaseStore
with TemporaryDirectory() as directory:
    store=CaseStore(Path(directory)/"jobs.db");store.enqueue("job")
    assert store.claim("job","attempt-a",0,seconds=10)
    assert store.claim("job","attempt-b",11)
    try:
        store.finish("job","attempt-a",12)
    except ValueError:
        print("Expired owner rejected")
    else:
        raise AssertionError("Old owner finished reclaimed work")


## Interpret the result

attempt-b owns it in this local exercise. External effects need their own identity and reconciliation arrangements.

The output belongs to this example. Check whether it is a stored value, a calculation, a check or a model reply before deciding what it establishes.


## Try independently

Which attempt owns the work after time eleven?

In the next cell, find `student_answer = "TODO"`. Replace only `"TODO"` with the answer and run the cell. Put text in quotes, leave numbers unquoted and write a yes or no answer as `True` or `False` with no quotes. The `assert` line checks the answer and displays a message if the check fails. Keep the worked solution below closed until the attempt is recorded. No attempt is reported as a pass.


In [ ]:
student_answer = "TODO"
if student_answer == "TODO":
    print("Not attempted yet.")
else:
    assert student_answer == 'attempt-b', "Revisit the supplied values and the worked explanation."
    print("This prediction matches the stated example.")


## Worked prediction and organisational connection

attempt-b owns it in this local exercise. External effects need their own identity and reconciliation arrangements.

Return to the written lesson and identify who can accept the contribution, which evidence needs correction and what remains outside the application's authority. If this is a business-design lesson, write a case-specific decision record rather than substituting the model's answer for that judgement.


In [ ]:
student_answer = "attempt-b"
assert student_answer == 'attempt-b'
